<a href="https://colab.research.google.com/github/Altaieb-Mohammed/lab_2corse/blob/master/lab2_3_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data

import pandas as pd
url = "https://raw.githubusercontent.com/Altaieb-Mohammed/lab_2corse/master/Bank_Customers.csv"
data = pd.read_csv(url)

# Basic information
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
#2 Calculate net estate
df['net_estate'] = df['estate'] - df['debts'] - df['wills']

# Check if distributions sum to net estate
df['total_distributed'] = df[['share_husband', 'share_wife', 'share_father',
                               'share_mother', 'share_sons', 'share_daughters']].sum(axis=1)

# Calculate difference
df['distribution_diff'] = df['net_estate'] - df['total_distributed']
print("\nDistribution Accuracy Check:")
print(f"Mean difference: {df['distribution_diff'].mean():.2f}")
print(f"Max difference: {df['distribution_diff'].max():.2f}")
print(f"Min difference: {df['distribution_diff'].min():.2f}")

In [ ]:
# 3Descriptive statistics for numerical columns
print("\nDescriptive Statistics - Estate and Debts:")
print(df[['estate', 'debts', 'wills', 'net_estate']].describe())

# Frequency of different family members
print("\nFrequency of Family Members Present:")
family_members = ['husband', 'wives', 'father', 'mother', 'sons', 'daughters',
                  'brothers_m', 'sisters_m', 'grandfather', 'grandmother']
for member in family_members:
    present_count = df[member].sum()
    percentage = (present_count / len(df)) * 100
    print(f"{member}: {present_count} cases ({percentage:.1f}%)")

In [ ]:
# Set up the plotting style
plt.style.use('seaborn-v0_8')

# 4Plot 1: Distribution of net estate
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.hist(df['net_estate'], bins=50, edgecolor='black', alpha=0.7)
plt.title('Distribution of Net Estate')
plt.xlabel('Net Estate Value')
plt.ylabel('Frequency')

# Plot 2: Most common family configurations
plt.subplot(1, 3, 2)
# Create a simple family configuration identifier
df['family_config'] = df[family_members].apply(lambda x: ''.join(x.astype(str)), axis=1)
top_configs = df['family_config'].value_counts().head(10)
top_configs.plot(kind='bar')
plt.title('Top 10 Family Configurations')
plt.xlabel('Family Configuration')
plt.ylabel('Count')
plt.xticks(rotation=45)

# Plot 3: Presence of key heirs
plt.subplot(1, 3, 3)
key_heirs = ['husband', 'wives', 'father', 'mother', 'sons', 'daughters']
presence_counts = [df[heir].sum() for heir in key_heirs]
plt.bar(key_heirs, presence_counts)
plt.title('Presence of Key Heirs')
plt.xlabel('Heir Type')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 5Correlation between estate size and heir shares
print("\nCorrelation between Net Estate and Heir Shares:")
correlation_matrix = df[['net_estate', 'share_husband', 'share_wife', 'share_father',
                         'share_mother', 'share_sons', 'share_daughters']].corr()
print(correlation_matrix['net_estate'].sort_values(ascending=False))

# Visualize correlations
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix: Estate vs Heir Shares')
plt.tight_layout()
plt.show()

LAB 3: PATTERN ANALYSIS AND HEIRSHIP RULES

In [ ]:
# 1Group by presence of key heirs
print("\n=== Inheritance Patterns by Family Composition ===")

# Case 1: When husband is present vs absent
husband_cases = df.groupby('husband').agg({
    'share_husband': 'mean',
    'share_wife': 'mean',
    'share_father': 'mean',
    'share_mother': 'mean',
    'share_sons': 'mean',
    'share_daughters': 'mean',
    'net_estate': 'mean'
}).round(2)

print("\n1. Husband Present (1) vs Absent (0):")
print(husband_cases)

# Case 2: When sons are present vs absent
print("\n2. Average Share Distribution by Sons Presence:")
sons_analysis = df.groupby('sons').agg({
    'share_sons': 'mean',
    'share_daughters': 'mean',
    'share_father': 'mean',
    'share_mother': 'mean'
}).round(2)
print(sons_analysis)

# Case 3: Father and Mother combinations
print("\n3. Father and Mother Presence Combinations:")
parent_combinations = df.groupby(['father', 'mother']).size().reset_index(name='count')
parent_combinations['percentage'] = (parent_combinations['count'] / len(df) * 100).round(1)
print(parent_combinations)

In [ ]:
# Calculate percentage shares
share_columns = ['share_husband', 'share_wife', 'share_father', 'share_mother',
                 'share_sons', 'share_daughters']

for col in share_columns:
    df[f'{col}_pct'] = (df[col] / df['net_estate']) * 100

# Get average percentages
print("\nAverage Shares as Percentage of Net Estate:")
avg_percentages = {}
for col in share_columns:
    pct_col = f'{col}_pct'
    # Only calculate for cases where the heir is present
    mask = df[col.replace('share_', '')] == 1
    avg_pct = df.loc[mask, pct_col].mean()
    avg_percentages[col] = avg_pct

for heir, pct in sorted(avg_percentages.items(), key=lambda x: x[1], reverse=True):
    print(f"{heir.replace('share_', '')}: {pct:.1f}%")

# Create visualization
plt.figure(figsize=(10, 6))
heirs = [h.replace('share_', '') for h in avg_percentages.keys()]
percentages = list(avg_percentages.values())

bars = plt.bar(heirs, percentages)
plt.title('Average Share as Percentage of Net Estate (When Present)')
plt.xlabel('Heir Type')
plt.ylabel('Average Percentage (%)')
plt.xticks(rotation=45)

# Add value labels on bars
for bar, pct in zip(bars, percentages):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{pct:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
print("\n=== Special Cases Analysis ===")

# 3Case A: Only sons and daughters (no parents, spouse)
case_a = df[(df['sons'] == 1) & (df['daughters'] == 1) &
            (df['father'] == 0) & (df['mother'] == 0) &
            (df['husband'] == 0) & (df['wives'] == 0)]

if len(case_a) > 0:
    print(f"\nA. Only Sons and Daughters: {len(case_a)} cases")
    print(f"   Average sons share: {case_a['share_sons'].mean():.2f}")
    print(f"   Average daughters share: {case_a['share_daughters'].mean():.2f}")
    print(f"   Sons:Daughters ratio: {(case_a['share_sons']/case_a['share_daughters']).mean():.2f}:1")

# Case B: Husband and wife only
case_b = df[(df['husband'] == 1) & (df['wives'] == 1) &
            (df['sons'] == 0) & (df['daughters'] == 0) &
            (df['father'] == 0) & (df['mother'] == 0)]

if len(case_b) > 0:
    print(f"\nB. Husband and Wife Only: {len(case_b)} cases")
    print(f"   Average husband share: {case_b['share_husband'].mean():.2f}")
    print(f"   Average wife share: {case_b['share_wife'].mean():.2f}")

# Case C: Father and mother only
case_c = df[(df['father'] == 1) & (df['mother'] == 1) &
            (df['sons'] == 0) & (df['daughters'] == 0) &
            (df['husband'] == 0) & (df['wives'] == 0)]

if len(case_c) > 0:
    print(f"\nC. Father and Mother Only: {len(case_c)} cases")
    print(f"   Average father share: {case_c['share_father'].mean():.2f}")
    print(f"   Average mother share: {case_c['share_mother'].mean():.2f}")

LAB 4: ADVANCED ANALYSIS AND PREDICTIVE INSIGHTS

In [ ]:
# 1Create meaningful heir combination categories
def categorize_heirs(row):
    if row['husband'] == 1 and row['wives'] == 0:
        base = "Husband"
    elif row['husband'] == 0 and row['wives'] == 1:
        base = "Wife"
    elif row['husband'] == 1 and row['wives'] == 1:
        base = "Husband+Wife"
    else:
        base = "NoSpouse"

    if row['sons'] == 1 and row['daughters'] == 1:
        children = "+Children(Both)"
    elif row['sons'] == 1:
        children = "+Sons"
    elif row['daughters'] == 1:
        children = "+Daughters"
    else:
        children = "+NoChildren"

    if row['father'] == 1 and row['mother'] == 1:
        parents = "+Parents(Both)"
    elif row['father'] == 1:
        parents = "+Father"
    elif row['mother'] == 1:
        parents = "+Mother"
    else:
        parents = "+NoParents"

    return f"{base}{children}{parents}"

df['heir_category'] = df.apply(categorize_heirs, axis=1)

# Analyze by category
print("\n=== Analysis by Heir Category ===")
category_stats = df.groupby('heir_category').agg({
    'net_estate': ['count', 'mean', 'median'],
    'share_husband': 'mean',
    'share_wife': 'mean',
    'share_father': 'mean',
    'share_mother': 'mean',
    'share_sons': 'mean',
    'share_daughters': 'mean'
}).round(2)

# Sort by frequency
category_stats = category_stats.sort_values(('net_estate', 'count'), ascending=False)
print(category_stats.head(10))

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

print("\n=== Regression Analysis: Predicting Sons' Share ===")

# Prepare data for predicting sons' share
X = df[['net_estate', 'husband', 'wives', 'father', 'mother', 'daughters']]
y = df['share_sons']

# 2Filter only cases where sons are present
mask = df['sons'] == 1
X_filtered = X[mask]
y_filtered = y[mask]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered, test_size=0.2, random_state=42
)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.4f}")
print("\nCoefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature}: {coef:.4f}")
print(f"  Intercept: {model.intercept_:.4f}")

In [ ]:
print("\n=== Inferred Inheritance Rules ===")

# 3Rule 1: Sons vs Daughters ratio
print("\n1. Sons vs Daughters Inheritance Ratio:")
son_daughter_cases = df[(df['sons'] == 1) & (df['daughters'] == 1)]
if len(son_daughter_cases) > 0:
    avg_ratio = (son_daughter_cases['share_sons'] / son_daughter_cases['share_daughters']).mean()
    print(f"   When both sons and daughters are present:")
    print(f"   Average ratio (Sons:Daughters) = {avg_ratio:.2f}:1")
    print(f"   This suggests sons receive approximately {avg_ratio:.1f} times what daughters receive")

# Rule 2: Spouse inheritance
print("\n2. Spouse Inheritance Patterns:")
husband_cases = df[df['husband'] == 1]
wife_cases = df[df['wives'] == 1]

if len(husband_cases) > 0:
    husband_pct = (husband_cases['share_husband'] / husband_cases['net_estate']).mean() * 100
    print(f"   Husband's average share: {husband_pct:.1f}% of net estate")

if len(wife_cases) > 0:
    wife_pct = (wife_cases['share_wife'] / wife_cases['net_estate']).mean() * 100
    print(f"   Wife's average share: {wife_pct:.1f}% of net estate")

# Rule 3: Parental inheritance
print("\n3. Parental Inheritance Patterns:")
father_cases = df[df['father'] == 1]
mother_cases = df[df['mother'] == 1]

if len(father_cases) > 0 and len(mother_cases) > 0:
    father_pct = (father_cases['share_father'] / father_cases['net_estate']).mean() * 100
    mother_pct = (mother_cases['share_mother'] / mother_cases['net_estate']).mean() * 100
    print(f"   Father's average share: {father_pct:.1f}% of net estate")
    print(f"   Mother's average share: {mother_pct:.1f}% of net estate")
    print(f"   Father:Mother ratio = {(father_pct/mother_pct):.2f}:1")

In [ ]:
# Create comprehensive summary
print("\n" + "="*60)
print("COMPREHENSIVE INHERITANCE ANALYSIS REPORT")
print("="*60)

print(f"\nDataset Overview:")
print(f"• Total cases: {len(df):,}")
print(f"• Average net estate: {df['net_estate'].mean():.2f}")
print(f"• Median net estate: {df['net_estate'].median():.2f}")
print(f"• Estate range: {df['net_estate'].min():.2f} to {df['net_estate'].max():.2f}")

print(f"\nMost Common Family Configurations:")
top_5_categories = df['heir_category'].value_counts().head(5)
for i, (category, count) in enumerate(top_5_categories.items(), 1):
    percentage = (count / len(df)) * 100
    print(f"{i}. {category}: {count} cases ({percentage:.1f}%)")

print(f"\nKey Findings:")
print("1. Distribution accuracy: ", end="")
if abs(df['distribution_diff'].mean()) < 1:
    print("Excellent (distributions sum correctly to net estate)")
else:
    print(f"Check needed (average difference: {df['distribution_diff'].mean():.2f})")

#4 Final visualization: Comparison of all heir shares
plt.figure(figsize=(12, 6))

# Prepare data for box plot
share_data = []
labels = []
for heir in ['husband', 'wife', 'father', 'mother', 'sons', 'daughters']:
    share_col = f'share_{heir}'
    # Only include cases where the heir is present
    mask = df[heir] == 1
    if mask.any():
        share_data.append(df.loc[mask, share_col])
        labels.append(heir.capitalize())

plt.boxplot(share_data, labels=labels)
plt.title('Distribution of Shares by Heir Type (When Present)')
plt.ylabel('Share Amount')
plt.xlabel('Heir Type')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()